# Generate Paper-Ready Tables and Plots

This notebook creates publication-ready visualizations and tables for the paper.

**UPDATED for new metric format**

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Load All Results

In [ ]:
# Load all experiment results
metrics_files = list(Path("results/raw").glob("*_metrics.json"))

results = []
for f in metrics_files:
    with open(f, "r", encoding="utf-8") as fp:
        m = json.load(fp)
    
    # Extract model name from experiment name
    exp_name = m.get("experiment_name", f.stem)
    if "mpnet" in exp_name.lower():
        model = "MPNet"
    elif "jina_v5" in exp_name.lower():
        model = "Jina v5"
    elif "qwen3" in exp_name.lower() or "qwen" in exp_name.lower():
        model = "Qwen3"
    elif "bge" in exp_name.lower():
        model = "BGE-M3"
    elif "e5" in exp_name.lower():
        model = "E5-large"
    else:
        model = "Unknown"
    
    results.append({
        "experiment": exp_name,
        "dataset": m.get("dataset_name", "N/A"),
        "model": model,
        "accuracy": m["accuracy"],
        "macro_f1": m["macro_f1"],
        "weighted_f1": m["weighted_f1"],
        "macro_precision": m.get("macro_precision", 0.0),
        "macro_recall": m.get("macro_recall", 0.0),
        "mean_confidence": m.get("mean_confidence", 0.0),
        "num_samples": m.get("num_samples", 0),
        "num_classes": len(set([k for k in m.get("classification_report", {}).keys() if k.isdigit()]))
    })

results_df = pd.DataFrame(results)
print(f"Loaded {len(results_df)} experiments")
results_df.head(10)

## 2. Main Results Table

In [ ]:
# Create main results table
main_table = results_df[[
    "dataset", "model", "num_classes",
    "accuracy", "macro_f1", "weighted_f1"
]].copy()

# Format percentages
for col in ["accuracy", "macro_f1", "weighted_f1"]:
    main_table[col] = (main_table[col] * 100).round(2)

# Sort by macro F1
main_table = main_table.sort_values("macro_f1", ascending=False)

print("\n=== Main Results Table ===")
print(main_table.to_string(index=False))

# Save as LaTeX
Path("results/tables").mkdir(parents=True, exist_ok=True)
latex_table = main_table.to_latex(index=False, float_format="%.2f")
with open("results/tables/main_results.tex", "w") as f:
    f.write(latex_table)

# Save as CSV
main_table.to_csv("results/tables/main_results.csv", index=False)

print("\nTable saved to results/tables/")

## 3. Model Comparison

In [ ]:
# Compare different models
model_df = results_df.groupby("model").agg({
    "accuracy": "mean",
    "macro_f1": "mean",
    "weighted_f1": "mean",
}).reset_index().sort_values("macro_f1", ascending=False)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_df))
width = 0.25

ax.bar(x - width, model_df["accuracy"], width, label='Accuracy')
ax.bar(x, model_df["macro_f1"], width, label='Macro F1')
ax.bar(x + width, model_df["weighted_f1"], width, label='Weighted F1')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Average Performance by Model')
ax.set_xticks(x)
ax.set_xticklabels(model_df["model"])
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
Path("results/plots").mkdir(parents=True, exist_ok=True)
plt.savefig("results/plots/model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n=== Average Performance by Model ===")
print(model_df.to_string(index=False))

## 4. Dataset Difficulty Analysis

In [ ]:
# Analyze performance vs number of classes
dataset_df = results_df.groupby(["dataset", "num_classes"]).agg({
    "accuracy": "mean",
    "macro_f1": "mean",
}).reset_index().sort_values("num_classes")

fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(dataset_df["num_classes"], dataset_df["macro_f1"], s=100, alpha=0.6)

# Add labels
for idx, row in dataset_df.iterrows():
    ax.annotate(row["dataset"], 
                (row["num_classes"], row["macro_f1"]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.set_xlabel("Number of Classes")
ax.set_ylabel("Average Macro F1")
ax.set_title("Performance vs Dataset Difficulty (Number of Classes)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/plots/difficulty_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Performance Heatmap (Model × Dataset)

In [ ]:
# Create pivot table for heatmap
pivot_df = results_df.pivot_table(
    index="model",
    columns="dataset",
    values="macro_f1",
    aggfunc="mean"
)

plt.figure(figsize=(14, 6))
sns.heatmap(pivot_df, annot=True, fmt=".3f", cmap="YlGnBu", 
            cbar_kws={'label': 'Macro F1'})
plt.title('Macro F1 Score Heatmap (Model × Dataset)')
plt.xlabel('Dataset')
plt.ylabel('Model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("results/plots/performance_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Confidence Score Analysis

In [ ]:
# Compare confidence scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# By model
conf_by_model = results_df.groupby("model")["mean_confidence"].mean().sort_values(ascending=False)
ax1.barh(range(len(conf_by_model)), conf_by_model.values)
ax1.set_yticks(range(len(conf_by_model)))
ax1.set_yticklabels(conf_by_model.index)
ax1.set_xlabel("Mean Confidence Score")
ax1.set_title("Average Confidence by Model")
ax1.set_xlim(0, 1)

# By dataset
conf_by_dataset = results_df.groupby("dataset")["mean_confidence"].mean().sort_values(ascending=False)
ax2.barh(range(len(conf_by_dataset)), conf_by_dataset.values)
ax2.set_yticks(range(len(conf_by_dataset)))
ax2.set_yticklabels(conf_by_dataset.index)
ax2.set_xlabel("Mean Confidence Score")
ax2.set_title("Average Confidence by Dataset")
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.savefig("results/plots/confidence_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Summary Statistics Table

In [ ]:
# Create summary statistics
summary = results_df[["accuracy", "macro_f1", "weighted_f1"]].describe()
summary = (summary * 100).round(2)

print("\n=== Summary Statistics (%) ===")
print(summary)

# Save
summary.to_csv("results/tables/summary_statistics.csv")
summary.to_latex("results/tables/summary_statistics.tex", float_format="%.2f")

## 8. Best Results Summary

In [ ]:
# Best performing configurations
print("\n=== Top 10 Experiments by Macro F1 ===")
top10 = results_df.nlargest(10, "macro_f1")[[
    "experiment", "dataset", "model", "num_classes",
    "accuracy", "macro_f1", "weighted_f1"
]].copy()

for col in ["accuracy", "macro_f1", "weighted_f1"]:
    top10[col] = (top10[col] * 100).round(2)

print(top10.to_string(index=False))

# Save
top10.to_csv("results/tables/top10_results.csv", index=False)
top10.to_latex("results/tables/top10_results.tex", index=False, float_format="%.2f")

## 9. Model Consistency Analysis

In [ ]:
# Analyze model consistency (std dev across datasets)
consistency = results_df.groupby("model").agg({
    "macro_f1": ["mean", "std", "min", "max"]
}).round(4)

consistency.columns = ['_'.join(col) for col in consistency.columns]
consistency = consistency.sort_values("macro_f1_mean", ascending=False)

print("\n=== Model Consistency (Macro F1) ===")
print(consistency)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

models = consistency.index
means = consistency["macro_f1_mean"]
stds = consistency["macro_f1_std"]

ax.bar(range(len(models)), means, yerr=stds, capsize=5)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models)
ax.set_ylabel("Macro F1")
ax.set_title("Model Performance Consistency (Mean ± Std)")
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("results/plots/model_consistency.png", dpi=300, bbox_inches="tight")
plt.show()

consistency.to_csv("results/tables/model_consistency.csv")

## 10. Export Complete Results

In [ ]:
# Export all results to Excel for easy viewing
with pd.ExcelWriter("results/tables/all_results.xlsx") as writer:
    results_df.to_excel(writer, sheet_name="All Results", index=False)
    main_table.to_excel(writer, sheet_name="Main Table", index=False)
    top10.to_excel(writer, sheet_name="Top 10", index=False)
    summary.to_excel(writer, sheet_name="Summary Stats")
    model_df.to_excel(writer, sheet_name="Model Comparison", index=False)
    consistency.to_excel(writer, sheet_name="Model Consistency")

print("\n✅ All tables and plots generated!")
print("📁 Tables saved to: results/tables/")
print("📊 Plots saved to: results/plots/")
print("\nGenerated files:")
print("  - all_results.xlsx (Complete results with multiple sheets)")
print("  - main_results.csv/.tex")
print("  - top10_results.csv/.tex")
print("  - summary_statistics.csv/.tex")
print("  - model_consistency.csv")
print("  - Various plots (PNG, 300 DPI)")